# Pembuktian Tiga Poin Notulensi KOZY

Notebook ini **menguji**, bukan menjelaskan. Tiap sel menghasilkan angka yang bisa dibantah.

| # | Pertanyaan notulensi | Yang diuji di sini |
|---|---|---|
| **1** | Random Forest / XGBoost / SVR itu peningkatan dari baseline linear regression, kan? | Uji silang berulang + uji-t berpasangan + kurva belajar |
| **2** | Poin 8 diganti pakai K-Prototypes | K-Prototypes vs K-Means one-hot, dinilai dengan metrik campuran yang sama |
| **3** | Segmentasi berdasarkan variabel (harga, fasilitas), bukan titik lokasi | Bukti kebocoran kalau harga ikut jadi variabel cluster |

**Data:** `pasar.json` — 307 iklan sewa (240 Mamikos + 67 Papikost), fasilitas utuh.

Bukan `kozy_dataset_final.csv`, karena di berkas itu fasilitas Mamikos kosong total — dibuktikan di sel berikutnya.

In [ ]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")   # redam peringatan hitung-core di Windows
import json, io, warnings, collections
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True)

# ── gaya grafik: satu aksen biru, sumbu & grid dibuat redup ──────────────
BIRU, JINGGA, TOSCA, KUNING = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
ABU, TINTA = "#9a9a97", "#0b0b0b"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.edgecolor": "#d8d8d4", "axes.linewidth": 0.8, "axes.grid": True, "grid.color": "#ececea",
    "grid.linewidth": 0.7, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": "#52514e", "ytick.color": "#52514e", "text.color": TINTA, "figure.facecolor": "white",
})
rp = lambda v: f"Rp {v:,.0f}".replace(",", ".")

# ── cari pasar.json di mana pun notebook ini dijalankan ─────────────────
KANDIDAT = ["web/src/data/pasar.json", "../web/src/data/pasar.json", "../../web/src/data/pasar.json",
            "src/data/pasar.json", r"C:\PROJECT ANALITIKA DATA\KOZY\web\src\data\pasar.json"]
JALUR = next((p for p in KANDIDAT if os.path.exists(p)), None)
assert JALUR, "pasar.json tidak ketemu — sesuaikan KANDIDAT"
P = json.load(io.open(JALUR, encoding="utf-8")); kos = P["kos"]
print(f"Sumber   : {P['meta']['sumber']}  |  {P['meta']['n']} baris  |  {P['meta']['n_sumber']}")

# ── kenapa bukan kozy_dataset_final.csv ─────────────────────────────────
for c in ["../scrapers/output/kozy_dataset_final.csv", r"C:\PROJECT ANALITIKA DATA\KOZY\KOZY-repo\scrapers\output\kozy_dataset_final.csv"]:
    if os.path.exists(c):
        b = pd.read_csv(c)
        n_m = (b["sumber"] == "Mamikos").sum()
        print(f"\nPemeriksaan kozy_dataset_final.csv ({len(b)} baris):")
        print(f"  fasilitas 'ac' terisi untuk Mamikos : {b.loc[b.sumber=='Mamikos','ac'].notna().sum()}/{n_m}   <- HILANG")
        print(f"  kategori kolom 'jenis'              : {b['jenis'].nunique()} (seharusnya 3) -> {sorted(b['jenis'].unique())}")
        break

# ── variabel apa saja yang BENAR-BENAR tersedia ─────────────────────────
FAS4 = ["km", "ac", "wifi", "kasur"]                 # dicatat kedua situs
FAS6 = FAS4 + ["kloset", "akses24"]                  # kloset & akses24 hanya Mamikos
KECS = sorted({k["kec"] for k in kos})
y    = np.array([k["harga"] for k in kos], float)

ketersediaan = pd.DataFrame([
    ("harga",                 sum(1 for k in kos if k.get("harga"))),
    ("jenis",                 sum(1 for k in kos if k.get("jenis"))),
    ("kecamatan",             sum(1 for k in kos if k.get("kec"))),
    ("fasilitas inti (4)",    len(kos)),
    ("kloset & akses24",      sum(1 for k in kos if k["sumber"] == "Mamikos")),
    ("dilihat (popularitas)", sum(1 for k in kos if k.get("dilihat"))),
    ("rating",                sum(1 for k in kos if k.get("rating"))),
    ("koordinat",             sum(1 for k in kos if k.get("lat"))),
], columns=["variabel", "terisi"])
ketersediaan["%"] = (ketersediaan.terisi / len(kos) * 100).round(0).astype(int)
ketersediaan["layak jadi variabel model/cluster?"] = np.where(ketersediaan["%"] >= 95, "ya", "TIDAK")
print(f"\nKetersediaan variabel (dari {len(kos)} baris):")
print(ketersediaan.to_string(index=False))

: 

---
## Poin 1 — Apakah RF/XGBoost/SVR benar-benar mengungguli baseline?

Agar adil, **model non-linear dituning dulu** (grid search), baru dibandingkan.
Semua model dilatih di skala `log(harga)` lalu dibalik, sehingga MAPE-nya sebanding.
Uji-t berpasangan memakai lipatan yang identik untuk semua model.

In [ ]:
from sklearn.model_selection import RepeatedKFold, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor

def fitur(k):
    v  = [1.0 if k["kec"] == c else 0.0 for c in KECS[1:]]
    v += [1.0 if f in k["fasilitas"] else 0.0 for f in FAS6]
    v += [1.0 if k["jenis"] == j else 0.0 for j in ("putri", "campur")]
    v += [1.0 if k["sumber"] == "Papikost" else 0.0]
    v += [np.log1p(k["dilihat"] or 0), 1.0 if k["dilihat"] else 0.0]
    return v

X, ly = np.array([fitur(k) for k in kos]), np.log(y)
dalam = KFold(5, shuffle=True, random_state=1)

grid = {
    "Random Forest": (RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
                      {"max_depth": [3, 5, None], "min_samples_leaf": [1, 3, 8], "max_features": ["sqrt", 0.6, 1.0]}),
    "XGBoost":       (XGBRegressor(n_estimators=300, random_state=0, verbosity=0, n_jobs=-1),
                      {"learning_rate": [0.02, 0.06], "max_depth": [2, 3, 5], "min_child_weight": [1, 5, 10]}),
    "SVR (RBF)":     (make_pipeline(StandardScaler(), SVR()),
                      {"svr__C": [0.3, 1, 3, 10], "svr__gamma": ["scale", 0.02], "svr__epsilon": [0.05, 0.1]}),
}
model = {"OLS log-linear (baseline)": LinearRegression(), "Ridge": RidgeCV(alphas=np.logspace(-3, 3, 25))}
for nama, (m, g) in grid.items():
    gs = GridSearchCV(m, g, cv=dalam, scoring="neg_mean_absolute_error", n_jobs=-1).fit(X, ly)
    model[nama] = gs.best_estimator_
    print(f"tuning {nama:14}: {gs.best_params_}")

# ── bandingkan pada lipatan yang SAMA ────────────────────────────────────
cv  = RepeatedKFold(n_splits=5, n_repeats=10, random_state=7)   # 50 lipatan: uji-t perlu banyak ulangan agar stabil
lip = list(cv.split(X))
skor = {}
for nama, m in model.items():
    s = []
    for tr, te in lip:
        m.fit(X[tr], ly[tr])
        p = np.exp(np.clip(m.predict(X[te]), 10, 18))
        s.append(np.mean(np.abs(y[te] - p) / y[te]) * 100)
    skor[nama] = np.array(s)

dasar = skor["OLS log-linear (baseline)"]
baris = []
for nama, arr in skor.items():
    d = arr - dasar
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d))) if nama != "OLS log-linear (baseline)" else np.nan
    baris.append({"model": nama, "MAPE %": round(arr.mean(), 2), "selisih vs baseline": round(d.mean(), 2) if not np.isnan(t) else None,
                  "t": round(t, 1) if not np.isnan(t) else None,
                  "kesimpulan": "(acuan)" if np.isnan(t) else ("lebih BAIK (signifikan)" if t < -2.5 else "lebih BURUK (signifikan)" if t > 2.5 else "tidak berbeda nyata")})
tabel = pd.DataFrame(baris).sort_values("MAPE %").reset_index(drop=True)
print("\n", tabel.to_string(index=False))

# ── kurva belajar: apakah data lebih banyak mengubah kesimpulan? ─────────
rng, kurva = np.random.default_rng(3), {k: [] for k in ["OLS", "Ridge", "RF", "XGB"]}
NS = [60, 100, 150, 200, 245]
for n in NS:
    tmp = {k: [] for k in kurva}
    for _ in range(20):
        idx = rng.permutation(len(y)); tr, te = idx[:n], idx[245:]
        for lab, m in [("OLS", model["OLS log-linear (baseline)"]), ("Ridge", model["Ridge"]),
                       ("RF", model["Random Forest"]), ("XGB", model["XGBoost"])]:
            m.fit(X[tr], ly[tr]); p = np.exp(np.clip(m.predict(X[te]), 10, 18))
            tmp[lab].append(np.mean(np.abs(y[te] - p) / y[te]) * 100)
    for k in kurva: kurva[k].append(np.mean(tmp[k]))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.15, 1]})
t2 = tabel.sort_values("MAPE %", ascending=False)
a1.barh(t2.model, t2["MAPE %"], color=[BIRU if "baseline" in m else ABU for m in t2.model], height=.62)
for i, (m, v) in enumerate(zip(t2.model, t2["MAPE %"])): a1.text(v + .06, i, f"{v:.2f}%", va="center", fontsize=8.5)
a1.axvline(dasar.mean(), color=BIRU, ls="--", lw=1, zorder=0)
a1.set_xlim(19, t2["MAPE %"].max() + .9); a1.set_xlabel("MAPE (makin kecil makin baik)")
a1.set_title("Setelah dituning adil, semuanya berimpit"); a1.grid(axis="y", visible=False)

for lab, warna in [("OLS", ABU), ("Ridge", BIRU), ("RF", JINGGA), ("XGB", TOSCA)]:
    a2.plot(NS, kurva[lab], marker="o", ms=4, lw=1.8, color=warna, label=lab)
    a2.text(NS[-1] + 4, kurva[lab][-1], lab, color=warna, fontsize=8.5, va="center", fontweight="bold")
a2.set_xlabel("jumlah data latih"); a2.set_ylabel("MAPE %"); a2.set_xlim(55, 275)
a2.set_title("Kurva sudah mendatar — tambah data tidak membuka jarak")
plt.tight_layout(); plt.show()

---
## Poin 2 — Apakah K-Prototypes lebih baik daripada K-Means?

Dua jebakan yang dihindari di sini:

1. **Menilai dengan metrik yang berbeda.** Silhouette K-Prototypes sering dihitung hanya di dimensi numerik — itu mengabaikan bagian kategorikal dan membuatnya terlihat unggul palsu. Di sini **kedua metode dinilai dengan matriks jarak campuran yang sama**.
2. **Mengabaikan stabilitas.** Cluster yang berubah-ubah tiap dijalankan tidak layak dipakai. Diuji dengan Adjusted Rand Index (ARI) pada subsampel 80%.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

def kprototypes(Xn, Xc, k, gamma, seed=0, iterasi=100):
    """Huang (1997): numerik pakai Euclid kuadrat, kategorikal pakai jumlah ketidakcocokan."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Xn), k, replace=False)
    Cn, Cc = Xn[idx].copy(), Xc[idx].copy()
    label = np.full(len(Xn), -1)
    for _ in range(iterasi):
        d = ((Xn[:, None, :] - Cn[None]) ** 2).sum(2) + gamma * (Xc[:, None, :] != Cc[None]).sum(2)
        baru = d.argmin(1)
        if (baru == label).all(): break
        label = baru
        for j in range(k):
            m = label == j
            if not m.any(): continue
            Cn[j] = Xn[m].mean(0)
            for c in range(Xc.shape[1]):
                v, h = np.unique(Xc[m, c], return_counts=True); Cc[j, c] = v[h.argmax()]
    return label, Cn, Cc

def susun(pakai_harga):
    num = [([np.log(k["harga"])] if pakai_harga else []) + [float(f in k["fasilitas"]) for f in FAS4] for k in kos]
    cat = [[k["kec"], k["jenis"]] for k in kos]
    return StandardScaler().fit_transform(np.array(num)), np.array(cat, dtype=object)

Xn, Xc = susun(True)
GAMMA  = 0.5 * Xn.std(0).mean()
D = np.sqrt(np.maximum(((Xn[:, None, :] - Xn[None]) ** 2).sum(2) + GAMMA * (Xc[:, None, :] != Xc[None]).sum(2), 0))
Xoh = np.hstack([Xn, np.array([[float(k["kec"] == c) for c in KECS] + [float(k["jenis"] == j) for j in ("putra","putri","campur")] for k in kos])])

sil, ari = [], []
for k in range(2, 8):
    lp, _, _ = kprototypes(Xn, Xc, k, GAMMA, seed=0)
    lk = KMeans(k, n_init=10, random_state=0).fit_predict(Xoh)
    sil.append({"k": k,
                "K-Prototypes": round(silhouette_score(D, lp, metric="precomputed"), 3),
                "K-Means one-hot": round(silhouette_score(D, lk, metric="precomputed"), 3),
                "n terkecil": int(np.bincount(lp, minlength=k).min())})
    r = [adjusted_rand_score(*[kprototypes(Xn[s], Xc[s], k, GAMMA, seed=z)[0] for z in (u, u + 99)])
         for u in range(12) for s in [np.random.default_rng(u).choice(len(kos), int(len(kos) * .8), replace=False)]]
    ari.append(round(float(np.mean(r)), 2))

MIN_ANGGOTA = 15        # cluster di bawah ~5% data tidak berguna sebagai label produk
sdf = pd.DataFrame(sil); sdf["selisih"] = (sdf["K-Prototypes"] - sdf["K-Means one-hot"]).round(3)
sdf["stabilitas (ARI)"] = ari
sdf["layak?"] = np.where(sdf["n terkecil"] >= MIN_ANGGOTA, "ya", "cluster terlalu kecil")
sdf = sdf[["k", "K-Prototypes", "K-Means one-hot", "selisih", "stabilitas (ARI)", "n terkecil", "layak?"]]
print("Silhouette dihitung dengan matriks jarak campuran yang SAMA untuk kedua metode:\n")
print(sdf.to_string(index=False))
print(f"\nSelisih rata-rata K-Prototypes vs K-Means: {sdf['selisih'].mean():+.3f}  -> praktis IMBANG")
# pilih k: cluster harus cukup besar, lalu silhouette tertinggi. ARI dilaporkan apa adanya.
layak = sdf[sdf["n terkecil"] >= MIN_ANGGOTA]
K_PILIH = int(layak.loc[layak["K-Prototypes"].idxmax(), "k"]) if len(layak) else 3
print(f"Stabilitas ARI seluruh k: {sdf['stabilitas (ARI)'].min():.2f}-{sdf['stabilitas (ARI)'].max():.2f}  -> SEDANG, tidak ada yang kokoh")
print(f"k terpilih = {K_PILIH}  (cluster terkecil >= {MIN_ANGGOTA} anggota, silhouette tertinggi di antaranya)")

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(sdf.k, sdf["K-Prototypes"],   marker="o", ms=5, lw=2, color=BIRU,   label="K-Prototypes")
ax.plot(sdf.k, sdf["K-Means one-hot"], marker="s", ms=5, lw=2, color=JINGGA, label="K-Means one-hot")
for _, r in sdf.iterrows():
    ax.annotate(f"{r['K-Prototypes']:.3f}", (r.k, r["K-Prototypes"]), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7.5, color=BIRU)
ax.set_xlabel("jumlah cluster (k)"); ax.set_ylabel("silhouette (metrik campuran)")
ax.set_title("Berimpit — keunggulan K-Prototypes bukan pada akurasi"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

---
## Poin 3 — Bolehkah harga menjadi variabel segmentasi?

Setuju bahwa segmentasi harus berbasis **karakteristik**, bukan titik koordinat.
Tetapi ada satu jebakan: kalau cluster dibentuk **memakai harga**, lalu cluster itu dipakai menilai
apakah harga wajar, penilaiannya **melingkar** — di aplikasi, harga itulah yang justru sedang dipertanyakan.

Diuji dengan menebak harga memakai median cluster, tiga cara.

In [ ]:
XnT, XcT = susun(False)                 # tanpa harga
GT = 0.5 * XnT.std(0).mean()
Xh = np.array([[1.0 if k["kec"] == c else 0.0 for c in KECS[1:]]
               + [1.0 if f in k["fasilitas"] else 0.0 for f in FAS6]
               + [1.0 if k["jenis"] == j else 0.0 for j in ("putri","campur")]
               + [1.0 if k["sumber"] == "Papikost" else 0.0] for k in kos])

hasil = {"Hedonic - tidak melihat harga kos yang dinilai": [],
         "Cluster PAKAI harga - mengintip jawabannya":      [],
         "Cluster TANPA harga - cara yang jujur":           []}
for tr, te in lip:
    m = LinearRegression().fit(Xh[tr], np.log(y[tr]))
    hasil["Hedonic - tidak melihat harga kos yang dinilai"].append(np.mean(np.abs(y[te] - np.exp(m.predict(Xh[te]))) / y[te]) * 100)
    for lab, (A, B, g) in [("Cluster PAKAI harga - mengintip jawabannya", (Xn, Xc, GAMMA)),
                           ("Cluster TANPA harga - cara yang jujur",      (XnT, XcT, GT))]:
        l, Cn, Cc = kprototypes(A[tr], B[tr], 4, g, seed=1)
        med = np.array([np.median(y[tr][l == j]) if (l == j).any() else np.median(y[tr]) for j in range(4)])
        d = ((A[te][:, None, :] - Cn[None]) ** 2).sum(2) + g * (B[te][:, None, :] != Cc[None]).sum(2)
        hasil[lab].append(np.mean(np.abs(y[te] - med[d.argmin(1)]) / y[te]) * 100)

bocor = pd.DataFrame([{"cara menebak harga": k, "MAPE %": round(np.mean(v), 1)} for k, v in hasil.items()])
print(bocor.to_string(index=False))
print("""
Baris 2 tampak paling bagus HANYA karena harga kos itu sendiri ikut dipakai menentukan clusternya.
Baris 3 adalah versi jujurnya, dan jelas kalah dari hedonic.
=> Clustering adalah alat MELABELI, bukan alat MENILAI HARGA.""")

# ── cluster tanpa harga: harga tetap terpisah sendiri ───────────────────
# k dipilih ulang pada konfigurasi TANPA harga - inilah yang akhirnya dipakai
pilih = []
for k in range(2, 8):
    l, _, _ = kprototypes(XnT, XcT, k, GT, seed=1)
    pilih.append({"k": k, "silhouette": round(silhouette_score(XnT, l), 3) if len(set(l)) > 1 else np.nan,
                  "n terkecil": int(np.bincount(l, minlength=k).min())})
pilih = pd.DataFrame(pilih)
pilih["layak?"] = np.where(pilih["n terkecil"] >= MIN_ANGGOTA, "ya", "cluster terlalu kecil")
sah = pilih[pilih["n terkecil"] >= MIN_ANGGOTA]
K_FINAL = int(sah.loc[sah["silhouette"].idxmax(), "k"]) if len(sah) else 3
print(f"\nPemilihan k pada konfigurasi tanpa harga (yang dipakai produksi):")
print(pilih.to_string(index=False))
print(f"k final = {K_FINAL}")

lab4, _, _ = kprototypes(XnT, XcT, K_FINAL, GT, seed=1)
prof = []
for j in sorted(set(lab4)):
    m = lab4 == j
    prof.append({"cluster": j, "n": int(m.sum()), "median harga": float(np.median(y[m])),
                 "p25": float(np.percentile(y[m], 25)), "p75": float(np.percentile(y[m], 75)),
                 **{f: round(float(np.mean([f in k["fasilitas"] for k in np.array(kos, dtype=object)[m]])) * 100) for f in FAS4}})
prof = pd.DataFrame(prof).sort_values("median harga", ascending=False).reset_index(drop=True)
prof["nama usulan"] = [f"Kelompok {i+1}" for i in range(len(prof))]
prof.loc[0, "nama usulan"] = "Kos Lengkap"; prof.loc[len(prof)-1, "nama usulan"] = "Kos Dasar"
print("\nProfil cluster yang dibentuk TANPA harga:\n")
print(prof.assign(**{"median harga": prof["median harga"].map(rp)}).drop(columns=["p25","p75"]).to_string(index=False))

fig, (b1, b2) = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.05]})
pos = np.arange(len(prof))
b1.barh(pos, prof["median harga"], color=BIRU, height=.6,
        xerr=[prof["median harga"] - prof.p25, prof.p75 - prof["median harga"]],
        error_kw=dict(ecolor="#b9c9e6", lw=4, capsize=0, alpha=.9))
b1.set_yticks(pos, [f"{r['nama usulan']}  (n={r.n})" for _, r in prof.iterrows()])
for i, v in enumerate(prof["median harga"]): b1.text(v * 1.03, i, rp(v), va="center", fontsize=8.5)
b1.invert_yaxis(); b1.set_xlim(0, prof.p75.max() * 1.35); b1.grid(axis="y", visible=False)
b1.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1e6:.1f} jt"))
b1.set_title("Harga terpisah rapi — padahal tidak ikut membentuk cluster")

M = prof[FAS4].values.astype(float)
b2.imshow(M, cmap="Blues", vmin=0, vmax=100, aspect="auto")
b2.set_xticks(range(len(FAS4)), ["KM dalam", "AC", "WiFi", "Kasur"])
b2.set_yticks(pos, prof["nama usulan"])
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        b2.text(j, i, f"{M[i,j]:.0f}%", ha="center", va="center", fontsize=8.5,
                color="white" if M[i, j] > 55 else TINTA)
b2.grid(False); b2.set_title("Yang membedakan cluster: fasilitasnya")
plt.tight_layout(); plt.show()

In [ ]:
AMBANG_T = 2.5          # |t| di atas ini baru disebut berbeda nyata
menang = tabel[(tabel["t"].notna()) & (tabel["t"] < -AMBANG_T)]["model"].tolist()
kalah  = tabel[(tabel["t"].notna()) & (tabel["t"] >  AMBANG_T)]["model"].tolist()
rentang_t = tabel["t"].dropna()

print("POIN 1 - RF/XGBoost/SVR sebagai peningkatan baseline?")
print(f"  model yang menang signifikan vs baseline : {menang or 'TIDAK ADA'}")
print(f"  model yang kalah signifikan              : {kalah or 'tidak ada'}")
print(f"  rentang t semua model                    : {rentang_t.min():+.1f} .. {rentang_t.max():+.1f}  (ambang {AMBANG_T})")
print(f"  jarak MAPE terbaik-terburuk              : {tabel['MAPE %'].max() - tabel['MAPE %'].min():.2f} poin")
print(f"  kedalaman pohon pilihan tuner            : RF max_depth={model['Random Forest'].max_depth}, XGB max_depth={model['XGBoost'].max_depth}")
print(f"  => {'TIDAK ADA yang mengungguli baseline secara meyakinkan.' if not menang else 'Ada yang menang: ' + ', '.join(menang)}")

print()
print("POIN 2 - K-Prototypes vs K-Means")
print(f"  selisih silhouette rata-rata : {sdf['selisih'].mean():+.3f}  (imbang)")
print(f"  stabilitas ARI semua k       : {sdf['stabilitas (ARI)'].min():.2f}-{sdf['stabilitas (ARI)'].max():.2f}  (sedang, tidak ada yang kokoh)")
print(f"  k saat membandingkan algoritma: {K_PILIH} (variabel lengkap)")
print(f"  k FINAL untuk produksi        : {K_FINAL} (tanpa harga, cluster terkecil >= {MIN_ANGGOTA})")
print("  => pilih K-Prototypes karena KETERBACAAN, bukan akurasi")

print()
print("POIN 3 - harga sebagai variabel cluster")
h, cp, ct = [round(np.mean(v), 1) for v in hasil.values()]
print(f"  hedonic {h}%  |  cluster PAKAI harga {cp}%  |  cluster TANPA harga {ct}%")
print(f"  selisih jujur vs hedonic : {ct - h:+.1f} poin -> clustering KALAH sebagai penilai harga")
print("  => cluster untuk MELABELI, hedonic untuk MEMVONIS")

---
## Kesimpulan

| Poin | Hasil uji | Rekomendasi |
|---|---|---|
| **1** | Setelah dituning adil, selisih MAPE semua model **di bawah 1 poin** dan tidak ada yang melewati ambang uji-t (lihat sel kesimpulan). Tuner memilih `max_depth` 2–3 — artinya tidak ada non-linearitas untuk dieksploitasi. Kurva belajar sudah mendatar, jadi menambah data sejenis tidak akan membuka jarak. | Jadikan **tabel benchmark** di skripsi sebagai bukti bahwa model sederhana sudah memadai — bukan model produksi. Pakai **hedonic (OLS/Ridge)** karena koefisiennya bisa dijelaskan ke pengguna ("AC menaikkan 58%"), sesuatu yang tidak bisa dilakukan XGBoost. |
| **2** | Silhouette K-Prototypes vs K-Means one-hot **berimpit** (selisih rata-rata di desimal ketiga). Stabilitas ARI tidak seragam — hanya sebagian k yang lolos 0,75. | **Pakai K-Prototypes**, tapi alasannya **keterbacaan** (centroid punya modus kategorikal yang bisa dibaca langsung), bukan akurasi. Pilih k dari kolom stabilitas, bukan dari silhouette tertinggi. Jangan klaim lebih akurat — nanti ditagih buktinya. |
| **3** | Cluster yang memakai harga tampak unggul (MAPE lebih rendah) **karena mengintip harga yang sedang dinilai**. Versi jujurnya kalah telak dari hedonic. | **Harga jangan jadi variabel cluster.** Cluster **melabeli** ("Kos Lengkap"), hedonic **memvonis** (wajar/mahal/murah). Label `highly recommended` = posisi kos terhadap harga wajar hedonic **di dalam** clusternya. |

### Catatan keterbatasan yang perlu disebut di sidang

- `rating` hanya terisi **9%**, `koordinat` **22%**, `popularitas` **79%** → tidak layak jadi variabel cluster. Rencana awal yang menyebut "rating" dan "popularitas" tidak bisa dieksekusi.
- MAPE ±21% sebagian besar **bukan kegagalan model**: kos dengan fitur identik pun harganya bisa berbeda 2,5–2,7× karena keputusan pemilik. Itu batas bawah yang tidak bisa ditembus dengan fitur yang ada.
- Justru **selisih** antara harga pemilik dan harga wajar itulah produk KOZY. Kalau error 0%, tidak akan pernah ada kos yang bisa disebut kemahalan.